# Training Wordle with GRPO — replicated **without** the Predibase SDK

This single notebook reproduces the two experiments from the course:

* **Lesson 7 — Calculating loss in GRPO.** Build the GRPO objective from scratch
  (reference vs. policy model, the probability ratio, PPO-style clipping, and the
  KL penalty) on a tiny LLM. *This lesson never used Predibase — it is reproduced
  essentially verbatim.*
* **Lesson 8 — Putting it all together: training Wordle.** The course launched a
  hosted GRPO job with `pb.finetuning.jobs.create(...)`. Here we run the **same
  dataset and the same three reward functions** using the open-source
  [**TRL `GRPOTrainer`**](https://huggingface.co/docs/trl) instead — no Predibase
  account, API key, or SDK required.

Everything runs on local / your-own hardware. Nothing here calls `predibase`.

---

### What maps to what

| Predibase (Lesson 8) | Open-source replacement here |
|---|---|
| `pb.finetuning.jobs.create(GRPOConfig(...))` | `trl.GRPOTrainer` + `trl.GRPOConfig` |
| `pb.datasets.from_pandas_dataframe(...)` | `datasets.load_dataset("predibase/wordle-grpo")` |
| `RewardFunctionsConfig(functions={...})` | list of Python callables passed to `reward_funcs=` |
| reward signature `f(prompt, completion, example)` | batched TRL signature `f(prompts, completions, **cols)` (adapter below) |
| `base_model="qwen2-5-7b-instruct"` | `"Qwen/Qwen2.5-7B-Instruct"` (with a smaller default for laptops) |
| automatic LoRA | explicit `peft.LoraConfig` passed to the trainer |
| `num_generations=16` | `GRPOConfig(num_generations=16)` |
| `SamplingParamsConfig(max_tokens=4096)` | `GRPOConfig(max_completion_length=4096)` |

> **Hardware.** GRPO needs a CUDA GPU. A 7B model with 16 generations wants a
> big card (A100/H100-class). The notebook exposes a **`QUICK`** switch that
> swaps in a small model + a data subset so you can smoke-test the whole
> pipeline on a single consumer GPU (or even verify wiring on CPU).

## 0. Install dependencies

`trl` brings in `transformers`, `accelerate`, `datasets` and `peft`. `vllm` is
optional but strongly recommended for fast GRPO generation on GPU.

In [ ]:
# Run once. Versions pinned to a known-good combo for GRPOTrainer.
%pip install -q "trl>=0.15.0" "transformers>=4.48.0" "peft>=0.14.0" \
                "datasets>=3.0.0" "accelerate>=1.2.0" pandas matplotlib

# peft>=0.14 raises if it finds an OLD torchao (e.g. Colab's 0.10.0):
#   ImportError: Found an incompatible version of torchao ...
# This notebook never uses torchao, so we simply remove it -> peft's
# is_torchao_available() returns False and the version guard is skipped.
# (If you'd rather keep it, replace the line below with:
#   %pip install -q -U "torchao>=0.16.0"   and then restart the runtime.)
%pip uninstall -y torchao

# TRL imports vLLM at module-load time. If an UNSUPPORTED vLLM is present
# (Colab ships 0.26.0; TRL supports 0.17.0-0.25.1) it can be built for a
# different CUDA and blow up with:
#   ImportError: libcudart.so.13: cannot open shared object file
# even before `use_vllm` is read. We remove it so `from trl import
# GRPOTrainer` succeeds and GRPO uses the built-in HF generation backend.
# To use vLLM instead, install a supported build matching your torch/CUDA:
#   %pip install -q "vllm>=0.17,<=0.25.1"   and set USE_VLLM = True below.
%pip uninstall -y vllm

# Optional (GPU only) — much faster rollouts. Pin a TRL-supported version,
# e.g.:  %pip install -q  vllm>=0.17,<=0.25.1   then set USE_VLLM = True.


> **If you already hit** `ImportError: Found an incompatible version of torchao`
> **or** `ImportError: libcudart.so.13: cannot open shared object file` (an
> unsupported vLLM built for another CUDA) earlier in this session: run the cell
> above, then **restart the runtime** (Colab: *Runtime → Restart session*) and
> re-run from the top. The bad modules are already loaded and only a restart
> clears them.

In [ ]:
import os, math, copy, textwrap
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

---
# Part 1 — Lesson 7: Calculating the GRPO loss from scratch

No Predibase here to begin with. We take a small language model, make a frozen
**reference** copy and a LoRA-adapted **policy**, and implement the GRPO
per-token loss step by step: the probability ratio, PPO clipping, and the KL
penalty.

Load a small instruct model to illustrate the loss mechanics. The default is
**`Qwen/Qwen2.5-3B-Instruct`**; the section is model-agnostic, so any causal LM
works.

> On a CPU-only or memory-tight machine, switch `MODEL_STR` to a smaller model
> such as `"Qwen/Qwen2.5-0.5B-Instruct"` or `"babylm/babyllama-100m-2024"`.
> Note that Qwen loads in **bfloat16** — the plotting helper below casts to
> float32 before converting to NumPy, since NumPy has no bf16 type.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_STR = "Qwen/Qwen2.5-3B-Instruct"   # small alternatives: "Qwen/Qwen2.5-0.5B-Instruct",
                                          # "babylm/babyllama-100m-2024" (runs on CPU)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_STR)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_STR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# pad on the left so we can append new tokens on the right
tokenizer.padding_side    = "left"
tokenizer.truncation_side = "left"


In [ ]:
prompt = "The quick brown fox jumped over the "

input_ids = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **input_ids,
        max_new_tokens=2,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated_portion = generated_text[len(prompt):]
print(f"Generated text: {prompt}\033[94m{generated_portion}\033[0m")

## Create reference and policy models

The **reference model** is the frozen base LLM. The **policy** is the same model
with a LoRA adapter whose weights are the only thing GRPO updates.

In [ ]:
from peft import LoraConfig, get_peft_model

# Frozen reference model (a deep copy so LoRA on the policy can't touch it)
ref_model = copy.deepcopy(base_model)

lora_config = LoraConfig(
    r=8,                       # rank of the update matrices
    lora_alpha=32,             # alpha scaling factor
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    init_lora_weights=False,   # random init so ratios != 1 for illustration
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)   # the policy
model.print_trainable_parameters()

## The policy-loss probability ratio

Three helpers: assemble a prompt+completion sequence with a mask marking the
generated tokens, compute per-token log-probabilities, and combine them into the
GRPO loss.

In [ ]:
def prepare_inputs(prompt, completion):
    prompt_tokens     = tokenizer(prompt, return_tensors="pt")
    completion_tokens = tokenizer(completion, return_tensors="pt")

    input_ids = torch.cat(
        [prompt_tokens["input_ids"], completion_tokens["input_ids"]], dim=1
    )
    attention_mask = torch.cat(
        [prompt_tokens["attention_mask"], completion_tokens["attention_mask"]], dim=1
    )

    prompt_length     = prompt_tokens["input_ids"].shape[1]
    completion_length = completion_tokens["input_ids"].shape[1]
    total_length      = prompt_length + completion_length

    # 1 on tokens the model generated, 0 on the prompt
    completion_mask = torch.zeros(total_length, dtype=torch.float32)
    completion_mask[prompt_length:] = 1.0
    return input_ids, attention_mask, completion_mask

In [ ]:
def compute_log_probs(model, input_ids, attention_mask):
    outputs = model(input_ids, attention_mask=attention_mask)
    log_probs = F.log_softmax(outputs.logits, dim=-1)
    # log-prob of the token actually present at each position
    return log_probs.gather(dim=-1, index=input_ids.unsqueeze(-1)).squeeze(-1)

In [ ]:
def grpo_loss(model, ref_model, prompt, completion, advantage):
    input_ids, attention_mask, completion_mask = prepare_inputs(prompt, completion)

    token_log_probs = compute_log_probs(model, input_ids, attention_mask)
    with torch.no_grad():
        ref_token_log_probs = compute_log_probs(ref_model, input_ids, attention_mask)

    # ratio = p_model / p_ref = exp(log p_model - log p_ref)
    ratio = torch.exp(token_log_probs - ref_token_log_probs)

    policy_loss = ratio * advantage
    per_token_loss = -policy_loss          # optimizers minimize; we maximize reward

    loss = (per_token_loss * completion_mask).sum() / completion_mask.sum()
    return loss

In [ ]:
grpo_loss(model, ref_model, prompt, "fence and", advantage=2.0)

At the very first optimization step the policy and reference are identical, so
every ratio is 1 and the loss reduces to the (negated) advantage:

In [ ]:
# model == ref_model  =>  ratio == 1  =>  loss == -advantage
grpo_loss(ref_model, ref_model, prompt, "fence and", advantage=2.0)

In [ ]:
completion = "fence and"
input_ids, attention_mask, completion_mask = prepare_inputs(prompt, completion)
with torch.no_grad():
    token_log_probs     = compute_log_probs(model,     input_ids, attention_mask)
    ref_token_log_probs = compute_log_probs(ref_model, input_ids, attention_mask)

ratio = torch.exp(token_log_probs - ref_token_log_probs)
print(ratio)

## Adding PPO-style clipping

Clipping the ratio to $[1-\epsilon,\,1+\epsilon]$ stops any single update from
moving the policy too far from the reference.

In [ ]:
def grpo_loss_with_clip(model, ref_model, prompt, completion, advantage, epsilon=0.2):
    input_ids, attention_mask, completion_mask = prepare_inputs(prompt, completion)

    token_log_probs = compute_log_probs(model, input_ids, attention_mask)
    with torch.no_grad():
        ref_token_log_probs = compute_log_probs(ref_model, input_ids, attention_mask)

    ratio = torch.exp(token_log_probs - ref_token_log_probs)

    unclipped = ratio * advantage
    clipped   = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantage
    policy_loss = torch.min(unclipped, clipped)

    per_token_loss = -policy_loss
    loss = (per_token_loss * completion_mask).sum() / completion_mask.sum()
    return loss

In [ ]:
grpo_loss_with_clip(model, ref_model, prompt, "fence and", advantage=2.0, epsilon=0.2)

Again, at step 1 (`model is ref_model`) clipping changes nothing — the loss is
still the advantage.

In [ ]:
grpo_loss_with_clip(ref_model, ref_model, prompt, "fence and", advantage=2.0)

### Visualising which tokens get clipped

The course imported `visualize_clipped_ratios` from a `utils` module. We
reimplement it inline so the notebook is self-contained.

In [ ]:
import matplotlib.pyplot as plt

def visualize_clipped_ratios(ratio_unclipped, ratio_clipped, epsilon):
    # .float() so bf16 tensors (e.g. Qwen loaded in bfloat16) convert to numpy
    ru = ratio_unclipped.detach().cpu().float().numpy().ravel()
    rc = ratio_clipped.detach().cpu().float().numpy().ravel()
    x = np.arange(len(ru))
    was_clipped = ~np.isclose(ru, rc)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhspan(1 - epsilon, 1 + epsilon, color="green", alpha=0.10,
               label=f"trust region [1±{epsilon}]")
    ax.axhline(1.0, color="gray", ls="--", lw=0.8)
    ax.bar(x - 0.2, ru, width=0.4, label="unclipped ratio", color="#4C72B0")
    ax.bar(x + 0.2, rc, width=0.4, label="clipped ratio",   color="#DD8452")
    for xi in x[was_clipped]:
        ax.annotate("clipped", (xi, max(ru[xi], rc[xi])),
                    textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=8, color="crimson")
    ax.set_xlabel("completion token index")
    ax.set_ylabel("ratio  p_policy / p_ref")
    ax.set_title(f"{was_clipped.sum()} / {len(ru)} output tokens were clipped")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
completion = "fence and"
input_ids, attention_mask, _ = prepare_inputs(prompt, completion)
with torch.no_grad():
    token_log_probs     = compute_log_probs(model,     input_ids, attention_mask)
    ref_token_log_probs = compute_log_probs(ref_model, input_ids, attention_mask)
    epsilon = 0.2
    ratio          = torch.exp(token_log_probs - ref_token_log_probs)
    ratio_clipped  = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)

# Slice off the prompt so we only plot the completion tokens. Compute the
# prompt length from the tokenizer instead of hard-coding it, so this works
# for any model (BabyLLama, Qwen, ...).
prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
visualize_clipped_ratios(ratio[0][prompt_len:], ratio_clipped[0][prompt_len:], epsilon)

## Adding the KL-divergence penalty

The KL term acts like a "gravitational pull" back toward the reference policy,
penalising the model for drifting too far.

In [ ]:
def grpo_loss_with_kl(model, ref_model, prompt, completion,
                      advantage, epsilon=0.2, beta=0.1):
    input_ids, attention_mask, completion_mask = prepare_inputs(prompt, completion)

    token_log_probs = compute_log_probs(model, input_ids, attention_mask)
    with torch.no_grad():
        ref_token_log_probs = compute_log_probs(ref_model, input_ids, attention_mask)

    ratio     = torch.exp(token_log_probs - ref_token_log_probs)
    unclipped = ratio * advantage
    clipped   = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantage
    policy_loss = torch.min(unclipped, clipped)

    # per-token KL(pi_ref || pi) = exp(-delta) + delta - 1
    delta        = token_log_probs - ref_token_log_probs
    per_token_kl = torch.exp(-delta) + delta - 1

    per_token_loss = -(policy_loss - beta * per_token_kl)
    loss = (per_token_loss * completion_mask).sum() / completion_mask.sum()
    return loss

In [ ]:
delta = np.linspace(-6, 6, 500)
kl_divergence = np.exp(-delta) + delta - 1

plt.figure(figsize=(8, 5))
plt.plot(delta, kl_divergence,
         label=r'$KL(\pi_{\mathrm{ref}} || \pi) = e^{-\Delta} + \Delta - 1$')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.fill_between(delta, kl_divergence, where=(delta > 0), color='red',   alpha=0.3,
                 label='Overconfident region (Δ > 0)')
plt.fill_between(delta, kl_divergence, where=(delta < 0), color='green', alpha=0.3,
                 label='Conservative region (Δ < 0)')
plt.title("KL Divergence as 'Gravitational Pull' Toward Reference Policy")
plt.xlabel(r'$\Delta = \log \pi - \log \pi_{\mathrm{ref}}$')
plt.ylabel('KL Penalty'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
for beta in [0, 0.1, 0.5]:
    loss = grpo_loss_with_kl(model, ref_model, prompt, "fence and",
                             advantage=2.0, epsilon=0.2, beta=beta)
    print(f"beta={beta}\nloss={loss.item():.3f}\n")

That is the complete GRPO per-token objective — ratio × advantage, clipped, with
a KL leash. TRL's `GRPOTrainer` (Part 2) implements exactly this loss; we now
just feed it a real task and real rewards.

---
# Part 2 — Lesson 8: Training Wordle with GRPO (Predibase-free)

Same dataset (`predibase/wordle-grpo`) and the same three reward functions as the
course. The only change is the training engine: **TRL `GRPOTrainer`** instead of
the hosted Predibase job.

### 2.1 Config switches

Set `QUICK = True` to smoke-test the whole pipeline quickly (small model, tiny
data subset). Set `QUICK = False` to reproduce the course setup
(`Qwen2.5-7B-Instruct`, 16 generations) — needs a large GPU.

In [ ]:
QUICK = True   # <-- flip to False for the faithful 7B / 16-generation run

if QUICK:
    BASE_MODEL      = "Qwen/Qwen2.5-0.5B-Instruct"  # tiny, runs on a modest GPU
    #                 "Qwen/Qwen2.5-3B-Instruct"    # mid-size: needs ~24GB VRAM
    NUM_GENERATIONS = 8
    MAX_COMPLETION  = 1024
    N_TRAIN         = 64            # subset of the dataset
    MAX_STEPS       = 20
else:
    BASE_MODEL      = "Qwen/Qwen2.5-7B-Instruct"    # the course's base model
    NUM_GENERATIONS = 16
    MAX_COMPLETION  = 4096
    N_TRAIN         = None          # full dataset
    MAX_STEPS       = -1            # let epochs decide

USE_VLLM = False   # vLLM speeds up rollouts on big GPUs, but Colab's build is
                   # often TRL-incompatible (see setup). Leave False unless you
                   # installed a supported vLLM (0.17-0.25.1) matching your torch.
print(f"BASE_MODEL={BASE_MODEL}  num_generations={NUM_GENERATIONS}  vllm={USE_VLLM}")

### 2.2 The reward functions

These are **identical** to the course's `reward_functions.py`. We write the file
out here so the notebook is self-contained and so we can `import` them exactly as
Lesson 8 did.

* `output_format_check` — rewards a well-formed `<think>…</think>\n<guess>WORD</guess>`
  whose guess is a valid 5-letter dictionary word.
* `uses_previous_feedback` — rewards guesses that respect earlier ✓ / - / x feedback.
* `guess_value` — rewards guesses that maximise expected information gain.

In [ ]:
%%writefile reward_functions.py
def output_format_check(prompt: str, completion: str, example: dict) -> int:
    import re
    import pandas as pd

    reward = 0
    try:
        # Add synthetic <think> as it's already part of the prompt and prefilled 
        # for the assistant to more easily match the regex
        completion = "<think>" + completion

        # Check if the format matches expected pattern:
        # <think> content </think> followed by <answer> content </answer>
        regex = (
            r"^<think>\s*([^<]*(?:<(?!/?think>)[^<]*)*)\s*<\/think>\n"
            r"<guess>\s*([\s\S]*?)\s*<\/guess>$"
        )

        # Search for the regex in the completion
        match = re.search(regex, completion, re.DOTALL)
        if match is None or len(match.groups()) != 2:
            return 0

        guess = match.groups()[1]
        guess = guess.strip()

        # If the word is not 5 characters, return 0
        if len(guess) != 5:
            return 0.1

        # Check if the guess is a valid word compared to a predifined list of words
        word_list = pd.read_csv(str(example["word_list"]))
        if guess not in word_list["Word"].values:
            return 0.5

        reward = 1.0
    except Exception:
        pass

    return reward


# Reward function that checks if the guess uses the previous feedback for its next guess
def uses_previous_feedback(prompt: str, completion: str, example: dict) -> int:
    import re
    import ast

    reward = 0
    try:
        # Add synthetic <think> as it's already part of the prompt and prefilled 
        # for the assistant to more easily match the regex
        completion = "<think>" + completion

        # Extract the guess from the completion
        regex = r"<guess>\s*([\s\S]*?)\s*<\/guess>$"
        match = re.search(regex, completion, re.DOTALL)
        if match is None or len(match.groups()) != 1:
            return 0

        guess = match.groups()[0].strip()
        if len(guess) != 5:
            return 0.0

        past_guess_history = ast.literal_eval(example["past_guess_history"])
        if len(past_guess_history) == 0:
            print("Uses previous feedback reward: 0.1 (No past guesses)")
            return 0.1

        correct_letter_to_position = {}
        valid_letter_to_position = {}
        wrong_letter_to_position = {}
        for _, past_feedback in past_guess_history:
            past_feedback = past_feedback.split(" ")
            for i, fb in enumerate(past_feedback):
                if '✓' in fb:
                    if fb[0] not in correct_letter_to_position:
                        correct_letter_to_position[fb[0]] = set()
                    correct_letter_to_position[fb[0]].add(i)
                elif '-' in fb:
                    if fb[0] not in valid_letter_to_position:
                        valid_letter_to_position[fb[0]] = set()
                    valid_letter_to_position[fb[0]].add(i)
                else:
                    if fb[0] not in wrong_letter_to_position:
                        wrong_letter_to_position[fb[0]] = set()
                    wrong_letter_to_position[fb[0]].add(i)

        for idx, letter in enumerate(guess):
            # Positive reward if guess reuses letter in confirmed correct position
            if (letter in correct_letter_to_position and idx in correct_letter_to_position[letter]):
                reward += 0.2
            # Reward if letter known to be in word is used in a new position
            elif (letter in valid_letter_to_position and idx not in valid_letter_to_position[letter]):
                reward += 0.1
            # Penalize reuse of known-in-word letter in same position (not exploring)
            elif (letter in valid_letter_to_position and idx in valid_letter_to_position[letter]):
                reward -= 0.2
            # Penalize use of known-absent letter
            elif letter in wrong_letter_to_position:
                reward -= 0.5
            else:
                # Reward unknown letters with partial credit for exploration
                reward += 0.05

    except Exception:
        return 0.0

    return reward


# Reward function that computes normalized information gain of the guess, i.e.,
# does the new guess reduce the uncertainty of the secret word the most
def guess_value(prompt: str, completion: str, example: dict) -> int:
    import math
    import re
    import ast
    import pandas as pd

    def validate_guess(secret: str, guess: str, raw_feedback: bool = False) -> str:
        feedback = []
        secret_list = list(secret)

        # Check for correct positions
        for i, (g_char, s_char) in enumerate(zip(guess, secret)):
            if g_char == s_char:
                feedback.append(f"{g_char}(✓) ")
                secret_list[i] = None
            else:
                feedback.append(None)

        # Check for misplaced letters
        for i, g_char in enumerate(guess):
            if feedback[i] is None:
                if g_char in secret_list:
                    feedback[i] = f"{g_char}(-) "
                    secret_list[secret_list.index(g_char)] = None
                else:
                    feedback[i] = f"{g_char}(x) "

        if raw_feedback:
            return feedback
        return "".join(feedback).strip()

    def filter_candidates(all_candidate_words, past_guesses):
        filtered = []
        for word in all_candidate_words:
            valid = True
            for past_guess, past_feedback in past_guesses:
                # Compute what the feedback would be if 'word' were the secret.
                candidate_feedback = validate_guess(word, past_guess)
                if candidate_feedback != past_feedback:
                    valid = False
                    break
            if valid:
                filtered.append(word)
        return filtered

    def compute_normalized_information_gain(all_candidate_words, past_guesses, guess):
        # First, filter the candidate words based on past guesses.
        candidates = filter_candidates(all_candidate_words, past_guesses)
        total_candidates = len(candidates)

        # If no candidates remain, return zeros.
        if total_candidates == 0:
            return 0.0, 0.0

        # Current uncertainty (entropy) before the guess.
        current_entropy = math.log2(total_candidates)

        # Partition candidates by the feedback pattern that would be produced by the current guess.
        feedback_groups = {}
        for word in candidates:
            # Get the raw feedback list (e.g., ['B(✓) ', 'R(✓) ', 'A(x) ', ...])
            feedback = validate_guess(word, guess, raw_feedback=True)
            # Create a simple representation for the feedback pattern.
            # '1' for correct position, '0' for wrong position, 'x' for letter not in word.
            feedback_pattern = "".join('1' if "✓" in fb else ('0' if "-" in fb else 'x') 
                                    for fb in feedback)
            feedback_groups.setdefault(feedback_pattern, []).append(word)

        expected_entropy = 0
        max_info_gain = 0
        # For each feedback group, compute its contribution to the expected entropy and the info gain.
        for group in feedback_groups.values():
            group_size = len(group)
            p = group_size / total_candidates
            # Entropy if this feedback is received.
            group_entropy = math.log2(group_size) if group_size > 0 else 0
            expected_entropy += p * group_entropy
            # Information gain for this feedback outcome.
            info_gain = current_entropy - group_entropy
            max_info_gain = max(max_info_gain, info_gain)

        # The expected gain is the reduction in entropy on average.
        expected_gain = current_entropy - expected_entropy

        # Normalize by the maximum possible gain, which is current_entropy (if you reduced to one candidate).
        normalized_expected_gain = expected_gain / current_entropy if current_entropy > 0 else 0
        normalized_max_gain = max_info_gain / current_entropy if current_entropy > 0 else 0

        return normalized_expected_gain, normalized_max_gain

    reward = 0
    try:
        # Add synthetic <think> as it's already part of the prompt and prefilled 
        # for the assistant to more easily match the regex
        completion = "<think>" + completion

        # Extract the guess from the completion
        regex = r"<guess>\s*([\s\S]*?)\s*<\/guess>$"
        match = re.search(regex, completion, re.DOTALL)
        if match is None or len(match.groups()) != 1:
            return 0

        guess = match.groups()[0].strip()
        if len(guess) != 5:
            return 0.0

        # Load the word list
        word_list = pd.read_csv(str(example["word_list"]))
        if guess not in word_list["Word"].values:
            return 0.0

        # Extract past guesses and feedback
        past_guess_history = ast.literal_eval(example["past_guess_history"])

        # Compute normalized information gain
        normalized_expected_gain, _ = compute_normalized_information_gain(
            word_list["Word"].values,
            past_guess_history,
            guess
        )

        # Compute reward based on normalized information gain
        reward = normalized_expected_gain
    except Exception:
        return 0.0

    return reward


In [ ]:
from reward_functions import (
    guess_value,
    output_format_check,
    uses_previous_feedback,
)
print("Imported reward functions:", [f.__name__ for f in
      (output_format_check, uses_previous_feedback, guess_value)])

### 2.3 Load the dataset

The same [`predibase/wordle-grpo`](https://huggingface.co/datasets/predibase/wordle-grpo)
dataset the course used. Each row carries the game-state `prompt` plus the extra
columns the reward functions need (`word_list`, `past_guess_history`, …).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("predibase/wordle-grpo", split="train")
print(dataset)
print("\nColumns:", dataset.column_names)
print("\nExample row (truncated):")
row = dataset[0]
for k, v in row.items():
    s = str(v)
    print(f"  {k:20s}: {s[:100]}{'…' if len(s) > 100 else ''}")

#### Cache the word list locally

The reward functions call `pd.read_csv(example["word_list"])` on **every**
scored completion. If that column is a URL, reading it repeatedly during training
is slow and needs constant network access. We download it once and rewrite the
column to point at a local file — the reward code is unchanged, it just reads a
fast local CSV.

In [ ]:
import urllib.request, functools

wl_value = str(dataset[0]["word_list"])
LOCAL_WORDLIST = os.path.abspath("wordle_word_list.csv")

if wl_value.startswith(("http://", "https://")):
    print("Downloading word list from:", wl_value)
    urllib.request.urlretrieve(wl_value, LOCAL_WORDLIST)
elif os.path.exists(wl_value):
    import shutil; shutil.copy(wl_value, LOCAL_WORDLIST)
else:
    # Column may already be a plain local path usable as-is.
    LOCAL_WORDLIST = wl_value

# sanity check + point every row at the cached copy
_wl = pd.read_csv(LOCAL_WORDLIST)
assert "Word" in _wl.columns, f"expected a 'Word' column, got {list(_wl.columns)}"
print(f"Word list: {len(_wl)} words, cached at {LOCAL_WORDLIST}")

dataset = dataset.map(lambda ex: {"word_list": LOCAL_WORDLIST})

In [ ]:
# Optional quick subset for QUICK mode
if N_TRAIN is not None:
    dataset = dataset.shuffle(seed=42).select(range(min(N_TRAIN, len(dataset))))
    print("Using subset of", len(dataset), "rows")

### 2.4 Adapt the reward functions to TRL's calling convention

Predibase calls each reward once per example: `f(prompt, completion, example)`.
TRL calls each reward once per **batch**: `f(prompts, completions, **columns)`
where every extra dataset column arrives as a list aligned with `completions`,
and the function returns a **list of floats**.

The adapter below bridges the two without touching the reward logic. It also
handles both plain-text and conversational (`[{"role","content"}]`) formats, and
— since recent TRL also injects non-column keyword args such as `trainer_state` —
it keeps only the kwargs that are real per-completion columns (lists the length
of `completions`), so it never tries to index a `TrainerState` object.
*(Unit-tested on synthetic Wordle data — plain and conversational, valid /
too-short / out-of-vocabulary guesses, and with a `trainer_state` extra present —
matching the single-example outputs.)*

In [ ]:
# Wrap a Predibase-style f(prompt, completion, example) reward into a
# batched TRL reward  f(prompts, completions, **columns) -> list[float].
def make_trl_reward(single_fn):
    def trl_fn(prompts, completions, **kwargs):
        n = len(completions)
        # TRL passes dataset columns (lists aligned to `completions`) AND some
        # extras that are NOT per-completion lists -- notably `trainer_state`
        # (a single TrainerState object). Keep only the real columns so we
        # never do v[i] on a non-indexable object.
        cols = {k: v for k, v in kwargs.items()
                if isinstance(v, (list, tuple)) and len(v) == n}
        rewards = []
        for i in range(n):
            p, c = prompts[i], completions[i]
            if isinstance(p, list):   # conversational prompt -> last turn's text
                p = p[-1]["content"]
            if isinstance(c, list):   # conversational completion -> assistant text
                c = c[0]["content"]
            example = {k: v[i] for k, v in cols.items()}
            try:
                rewards.append(float(single_fn(p, c, example)))
            except Exception:
                rewards.append(0.0)
        return rewards
    trl_fn.__name__ = single_fn.__name__   # nice names in the training logs
    return trl_fn

reward_funcs = [
    make_trl_reward(output_format_check),
    make_trl_reward(uses_previous_feedback),
    make_trl_reward(guess_value),
]

In [ ]:
# Quick self-check of the adapter on a real dataset row before training
_ex = dataset[0]
_demo_completion = "I'll open with a vowel-rich word.</think>\n<guess>CRANE</guess>"
_cols = {k: [dataset[0][k]] for k in dataset.column_names if k != "prompt"}
for rf in reward_funcs:
    print(f"{rf.__name__:24s}", rf(prompts=[_ex['prompt']],
                                   completions=[_demo_completion], **_cols))

### 2.5 Configure and launch GRPO

`GRPOConfig` mirrors the Predibase `GRPOConfig`: `num_generations`,
`max_completion_length` (= Predibase `SamplingParamsConfig.max_tokens`), and LoRA
(Predibase applied it automatically; here we pass an explicit `LoraConfig`, the
same target modules the course used for its SFT run).

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "down_proj", "up_proj"],
)

grpo_args = GRPOConfig(
    output_dir="wordle-grpo",
    # ---- mirrors the Predibase GRPOConfig ----
    num_generations=NUM_GENERATIONS,          # Predibase: num_generations
    max_completion_length=MAX_COMPLETION,     # Predibase: SamplingParamsConfig(max_tokens=...)
    # ---- standard GRPO / trainer knobs ----
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    beta=0.04,                                # KL coefficient (Part 1's `beta`)
    epsilon=0.2,                              # clip range (Part 1's `epsilon`)
    use_vllm=USE_VLLM,
    report_to="none",
)

trainer = GRPOTrainer(
    model=BASE_MODEL,
    reward_funcs=reward_funcs,   # <-- our three adapted rewards
    args=grpo_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

In [ ]:
# This is the open-source equivalent of pb.finetuning.jobs.create(...).
# Requires a CUDA GPU; on a large card with QUICK=False it reproduces the course run.
trainer.train()

In [ ]:
# Save the trained LoRA adapter (analogue of a Predibase checkpoint)
trainer.save_model("wordle-grpo/final")
print("Saved LoRA adapter to wordle-grpo/final")

### 2.6 Play Wordle with the trained model

Load the base model + trained adapter and let it produce a guess for a game
state, then score that guess with the same reward functions used in training.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
mdl = PeftModel.from_pretrained(mdl, "wordle-grpo/final")
mdl.eval()

sample = dataset[0]
prompt_text = sample["prompt"]
inputs = tok(prompt_text, return_tensors="pt").to(mdl.device)
with torch.no_grad():
    # Greedy decoding to judge the model's *best* guess (deterministic).
    # Switch to do_sample=True, temperature=0.7 to see sampled variety.
    out = mdl.generate(**inputs, max_new_tokens=256, do_sample=False,
                       pad_token_id=tok.eos_token_id)
completion_text = tok.decode(out[0][inputs["input_ids"].shape[1]:],
                             skip_special_tokens=True)
print("PROMPT (tail):\n", prompt_text[-400:])
print("\nCOMPLETION:\n", completion_text)

cols = {k: [sample[k]] for k in dataset.column_names if k != "prompt"}
print("\nReward breakdown:")
for rf in reward_funcs:
    print(f"  {rf.__name__:24s}", rf(prompts=[prompt_text],
                                     completions=[completion_text], **cols)[0])

---
# Part 3 — SFT and SFT → GRPO (Predibase-free)

Lesson 8 also showed (as commented code) how to warm-start with **supervised
fine-tuning** on `predibase/wordle-sft`, then continue from that checkpoint with
GRPO. Here are the open-source equivalents using `trl.SFTTrainer`.

### 3.1 SFT with TRL

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_dataset = load_dataset("predibase/wordle-sft", split="train")
if N_TRAIN is not None:
    sft_dataset = sft_dataset.shuffle(seed=42).select(range(min(N_TRAIN, len(sft_dataset))))
print(sft_dataset)

sft_peft = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "down_proj", "up_proj"],
)

sft_args = SFTConfig(
    output_dir="wordle-sft",
    num_train_epochs=(1 if QUICK else 10),   # course used epochs=10
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=1,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=BASE_MODEL,
    args=sft_args,
    train_dataset=sft_dataset,
    peft_config=sft_peft,
)
# sft_trainer.train()                       # uncomment to run
# sft_trainer.save_model("wordle-sft/final")

### 3.2 Continue from the SFT checkpoint with GRPO

Predibase expressed this with `continue_from_version="wordle/1"`. In TRL you load
the SFT LoRA checkpoint as the starting model, then run GRPO on top (this mirrors
the course's second config: `epochs=3`, `num_generations=8`,
`enable_early_stopping=False`).

In [ ]:
# from peft import PeftModel
#
# # Start GRPO from the SFT-tuned model instead of the raw base model
# sft_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL,
#               dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
# sft_model = PeftModel.from_pretrained(sft_base, "wordle-sft/final", is_trainable=True)
#
# grpo_args_stage2 = GRPOConfig(
#     output_dir="wordle-sft-then-grpo",
#     num_generations=8,                 # course: num_generations=8
#     max_completion_length=MAX_COMPLETION,
#     num_train_epochs=3,                # course: epochs=3
#     per_device_train_batch_size=8,
#     learning_rate=1e-5, beta=0.04, epsilon=0.2,
#     use_vllm=USE_VLLM, report_to="none",
# )
#
# trainer2 = GRPOTrainer(
#     model=sft_model,
#     reward_funcs=reward_funcs,
#     args=grpo_args_stage2,
#     train_dataset=dataset,
# )
# trainer2.train()
# trainer2.save_model("wordle-sft-then-grpo/final")
print("SFT -> GRPO recipe ready (uncomment to run).")

---
# Part 4 — Notes & gotchas

* **What was removed.** Every `from predibase import ...`, the `Predibase(...)`
  client, `pb.datasets.*`, `pb.repos.*`, and `pb.finetuning.jobs.create(...)`.
  No API key or account is needed.

* **Reward functions are untouched.** The three functions run verbatim; only a
  thin batched adapter wraps them for TRL. The adapter was validated on synthetic
  Wordle rows (plain + conversational; valid / too-short / out-of-dictionary
  guesses).

* **The `<think>` prefill.** The dataset prompts end with a prefilled `<think>`
  tag, so the model's completion begins *after* it. That is exactly why each
  reward function starts with `completion = "<think>" + completion` — this still
  holds under TRL, so the logic is correct unchanged.

* **`word_list` performance.** Reward functions re-read the word-list CSV on every
  scored completion. We cache it to a local file once; for very large runs you can
  additionally memoise `pd.read_csv` with `functools.lru_cache` for another speedup.

* **Config translation.** `num_generations` → `num_generations`;
  `SamplingParamsConfig(max_tokens=N)` → `max_completion_length=N`; automatic
  Predibase LoRA → explicit `LoraConfig`; the KL/clip terms from Part 1 map to
  `GRPOConfig(beta=..., epsilon=...)`.

* **Hardware reality.** GRPO on 7B with 16 generations is heavy — use a big GPU,
  `use_vllm=True`, and consider 4-bit loading (`BitsAndBytesConfig`) or DeepSpeed
  ZeRO. `QUICK=True` verifies the wiring end-to-end on far less.

* **Determinism.** As the course notes, sampling is stochastic — reward curves and
  final guesses will vary run to run.